In [ ]:
# V5 MIGRATION SAFETY STOP — remove ONLY once all dependencies are V5-native and verified.
raise RuntimeError('V5 MIGRATION INCOMPLETE: legacy inputs and/or intermediate outputs are still referenced. See V5/README.md and V5/DEPENDENCY_AUDIT.md. Do not run until the V5 path audit and data manifest pass.')


# Musanga V4 — NB02: paired RGB experiment (circles only versus circles + plots), v4_007

**Stage 1–2: CPU-only reference integrity, RGB ownership, paired pixel-label consistency, and reproducible candidate manifest.** First make a *single* authoritative split and training sample manifest for both RGB experiments, inspect coverage/contradictions, and then authorize GPU training in a subsequent NB02 revision. This notebook deliberately contains **no fitting**: it stops before GPU costs, following the earlier V4 debugging experience.

**Primary comparison:** Model A = original KVO-final exhaustive circles + site-specific partial tree annotations; Model B = *identical* inputs plus 78 Nadine exhaustive plot units (respecting their 50-m spatial grouping). All 30 retained circles and 78 plots are candidate outer-fold evaluation units; both models use the exact same held-out fold and assessment support. The historical V3.1 30-circle folds remain separately available as a retrospective benchmark, not interchangeable with this redesigned fold structure.

**Sources:** V3.1 audited reference and original ResNet18 U-Net recipe; V4 NB00 78 annotated plots and 3,324 crowns; V4 NB01 v4_002 frozen 50-m grouped / cover-balanced folds. Strictly distinguish mapped crown cover from field basal area.

**Policies:** Do not treat unknown pixels as negative. Within a circle, use KVO-final exhaustive Musanga polygons; within a non-circle plot, use Nadine exhaustive Musanga polygons. Negative non-Musanga tree crowns can also inform partial supervision. When the two reference systems disagree in overlapping areas, mask conflict pixels and report their geometry. Do not count intersecting references as independent validation replicates. For every fold, exclude *full training windows* touching held-out polygon + buffer, across all mosaics and all annotation sources. RGB training must not require DSM/MS coverage.


## 00 — Inputs, frozen split contract and reference provenance

Reads only metadata and vector files. Requires NB00 v4_001 Section 09 and NB01 v4_002 Sections 00–03 executed successfully. Input files and previous outputs are read-only.


In [ ]:
%pip -q install geopandas pyogrio shapely pandas numpy matplotlib rasterio
from pathlib import Path
import json,math,hashlib
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.strtree import STRtree
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                        Path('/content/drive/MyDrive/Musanga/Final')] if p.exists()),None)
assert ROOT is not None
DATA=ROOT/'Data'
OLD=ROOT/'Outputs/V3_1'
AUDIT=ROOT/'Outputs/v4/00_input_audit'
SPLIT_ROOT=ROOT/'Outputs/v4/01_spatial_cover_folds_v2'
OUT=ROOT/'Outputs/v4/02_rgb_circles_plus_plots_v3'
OUT.mkdir(parents=True,exist_ok=True)
V31=json.loads((OLD/'00_input_audit/00_input_manifest.json').read_text())
F78=json.loads((AUDIT/'00_v4_field78_contract.json').read_text())
SPLIT_CONTRACT=json.loads((SPLIT_ROOT/'01_reference_fold_contract_v2.json').read_text())
FOLD_CSV=Path(SPLIT_CONTRACT['output_csv'])
assert FOLD_CSV.exists(),FOLD_CSV
assert hashlib.sha256(FOLD_CSV.read_bytes()).hexdigest()==SPLIT_CONTRACT['csv_sha256'], 'NB01 fold file changed'
SPLITS=pd.read_csv(FOLD_CSV)
assert len(SPLITS)==108 and set(SPLITS.unit_type)=={'circle','plot'}
assert (SPLITS.unit_type=='circle').sum()==30 and (SPLITS.unit_type=='plot').sum()==78
assert SPLITS.groupby('spatial_group').fold.nunique().eq(1).all()
assert set(SPLITS.fold)=={0,1,2}
GEOMS=gpd.read_file(SPLIT_ROOT/'01_reference_groups_and_folds_v2.gpkg').to_crs('EPSG:32635')
assert len(GEOMS)==108
assert not GEOMS.duplicated(['unit_type','unit_id']).any()
SITES=['Baego','MOS','Yangambi','Yoko']
assert set(GEOMS.site)==set(SITES)
PLOT_PATH=Path(F78['plot_file'])
CROWN_PATH=Path(F78['exhaustive_plot_crowns'])
EXH_PATH=Path(F78['circle_positive_crowns'])
CIRCLE_PATH=Path(F78['circle_complete_roi'])
ROI_PATH=Path(V31['roi'])
for p in [PLOT_PATH,CROWN_PATH,EXH_PATH,CIRCLE_PATH,ROI_PATH]:assert p.exists(),p
print('REFERENCE:',len(GEOMS),'units;',GEOMS.spatial_group.nunique(),'spatial groups;',
      GEOMS.groupby('fold').size().to_dict(),'units by outer fold')
print('SPLIT FINGERPRINT:',SPLIT_CONTRACT['csv_sha256'][:20])
print('V3.1 KVO polygons:',EXH_PATH.name,'| V4 Nadine crowns:',CROWN_PATH.name)
print('OUT (new only):',OUT)


## 01 — Source annotation geometry, reference priority, and overlap conflict audit

**Source semantics:** V3.1 final exhaustive circle polygons are Musanga positives; inside an exhaustive circle the remaining area is certified non-Musanga. The V4 crown file has `Class_ID == 1` as Musanga; other classes are explicit negative crowns; the audited plot field has `annotated = True`. The latter does **not** by itself prove all unannotated background is biologically negative, so the assumption is recorded prominently for verification.

Compute overlap disagreements (positive in circle versus negative in plot, and vice versa) *by actual polygon area*, without reading large RGB mosaics. **Conflict pixels must be masked for Model B**, not silently relabelled. The circle reference remains the authoritative label for overlapping reference areas only where there is no contradiction.


In [ ]:
# NB02.01 — Topology-safe vector audit; fail closed on material geometry changes.
from shapely.validation import explain_validity
from shapely import make_valid,union_all,set_precision
from shapely.geometry import GeometryCollection
from shapely.ops import unary_union
REF_CRS='EPSG:32635'
circles_full=gpd.read_file(CIRCLE_PATH).to_crs(REF_CRS)
circles_full['unit_id']=circles_full.index.astype(str)
circles=circles_full[circles_full.unit_id.isin(
    GEOMS.loc[GEOMS.unit_type.eq('circle'),'unit_id'])].copy()
plots=gpd.read_file(PLOT_PATH).to_crs(REF_CRS)
plots['unit_id']=plots.Plot_name.astype(str)
assert len(circles)==30 and len(plots)==78
assert plots.unit_id.is_unique and circles.unit_id.is_unique

def polygon_parts(geom):
    if geom is None or geom.is_empty:return []
    if geom.geom_type in ('Polygon','MultiPolygon'):return [geom]
    return [part for child in getattr(geom,'geoms',[]) for part in polygon_parts(child)]

QA02=[]
def repair_polygon_geoms(geoms,source,maximum_relative_change=.05):
    fixed=[]
    for ix,geom in enumerate(geoms):
        if geom is None or geom.is_empty:
            QA02.append(dict(source=source,row=ix,status='EMPTY_STOP',relative_area_change=np.nan,
                             original_valid=False,reason='empty'))
            raise ValueError(f'{source} polygon {ix} is empty; cannot silently drop reference')
        original_valid=bool(geom.is_valid)
        reason='valid' if original_valid else explain_validity(geom)
        candidate=geom if original_valid else make_valid(geom)
        parts=polygon_parts(candidate)
        if not parts:
            QA02.append(dict(source=source,row=ix,status='NO_POLYGON_STOP',relative_area_change=np.nan,
                             original_valid=original_valid,reason=reason))
            raise ValueError(f'{source} {ix} cannot be repaired into polygons: {reason}')
        try:
            candidate=unary_union(parts) if len(parts)>1 else parts[0]
        except Exception as exc:
            raise ValueError(f'Failed polygon repair for {source}, row {ix}: {exc}') from exc
        if not candidate.is_valid or candidate.is_empty:
            raise ValueError(f'Invalid repaired geometry: {source} row {ix}')
        # Invalid input polygon area is GEOS-defined but may be ambiguous; audit both.
        original_area=float(geom.area)
        new_area=float(candidate.area)
        change=abs(new_area-original_area)/max(abs(new_area),abs(original_area),1.e-9)
        status='unchanged' if original_valid else 'repaired'
        QA02.append(dict(source=source,row=ix,status=status,
            relative_area_change=change,original_valid=original_valid,reason=reason))
        if change>maximum_relative_change:
            raise ValueError(
                f'{source} row {ix}: repair changes polygon area by {change:.2%} '
                f'(limit {maximum_relative_change:.0%}). Review original shape manually.')
        fixed.append(candidate)
    return fixed

def robust_union02(geoms,source):
    geoms=[q for q in geoms if q is not None and not q.is_empty]
    if not geoms:return GeometryCollection()
    try:
        out=unary_union(geoms)
    except Exception as exc:
        print(f'WARNING {source}: exact union failed ({type(exc).__name__}); '
              'retrying 1-cm precision grid after individual geometry validation.')
        try:
            out=union_all(geoms,grid_size=.01)
        except Exception as exc2:
            raise RuntimeError(f'{source}: union failed at exact and 1-cm precision. '
                               'Review topological conflicts before training.') from exc2
    if not out.is_valid:
        raise ValueError(f'{source}: union is invalid; do not use for labels')
    return out

# Check source polygons BEFORE attempting any union. This is the fix for the
# TopologyException in the raw KVO exhaustive-positive shapefile.
exh=gpd.read_file(EXH_PATH).to_crs(REF_CRS)
exh.geometry=repair_polygon_geoms(exh.geometry,'V3_1_exhaustive_Musanga')
crowns=gpd.read_file(CROWN_PATH).to_crs(REF_CRS)
crowns['is_musanga']=pd.to_numeric(crowns.Class_ID,errors='raise').eq(1)
crowns.geometry=repair_polygon_geoms(crowns.geometry,'V4_Nadine_plot_crowns')
circles.geometry=repair_polygon_geoms(circles.geometry,'V3_1_circle_boundaries')
plots.geometry=repair_polygon_geoms(plots.geometry,'V4_plot_boundaries')
print('EXHAUSTIVE REFERENCE REPAIR SUMMARY (no rows dropped):')
display(pd.DataFrame(QA02).groupby(['source','status']).agg(
    n=('row','size'),max_rel_area_change=('relative_area_change','max')).round(5))
pd.DataFrame(QA02).to_csv(OUT/'02_source_geometry_repairs.csv',index=False)
print('Saved per-polygon repair audit:',OUT/'02_source_geometry_repairs.csv')

# Partial crown data can be quarantined under a separate, documented policy:
# a partial annotation is OPTIONAL supplementary supervision; a retained exhaustive
# circle or plot is NOT optional and must continue to fail closed on material repairs.
# Keep source-row indices intact so exclusions are traceable.
partial={}
partial_quarantine=[]
for site,path in V31['shared_annotations'].items():
    v=gpd.read_file(path).to_crs(REF_CRS)
    assert 'Class_ID' in v.columns
    v['musanga']=pd.to_numeric(v.Class_ID,errors='coerce').eq(1)
    retained=[]
    for source_row,geom in zip(v.index.tolist(),v.geometry.tolist()):
        label=f'V3_1_partial_{site}'
        original_valid=(geom is not None and not geom.is_empty and bool(geom.is_valid))
        reason=('valid' if original_valid else
                'empty' if geom is None or geom.is_empty else explain_validity(geom))
        change=np.nan
        try:
            if geom is None or geom.is_empty:
                raise ValueError('empty crown')
            candidate=geom if original_valid else make_valid(geom)
            parts=polygon_parts(candidate)
            if not parts:
                raise ValueError('no polygonal component after make_valid')
            candidate=unary_union(parts) if len(parts)>1 else parts[0]
            if candidate.is_empty or not candidate.is_valid:
                raise ValueError('invalid candidate after repair')
            change=abs(float(candidate.area)-float(geom.area))/max(
                float(candidate.area),float(geom.area),1e-9)
            if change>.05:
                raise ValueError(f'area change {change:.2%} exceeds 5%')
            retained.append((int(source_row),candidate))
            QA02.append(dict(source=label,row=int(source_row),
                status='unchanged' if original_valid else 'repaired',
                relative_area_change=change,original_valid=original_valid,reason=reason))
        except Exception as exc:
            QA02.append(dict(source=label,row=int(source_row),
                status='quarantined_partial',relative_area_change=change,
                original_valid=original_valid,reason=f'{reason}; {exc}'))
            partial_quarantine.append(dict(site=site,source_row=int(source_row),
                relative_area_change=change,original_valid=original_valid,
                reason=str(exc),source_path=str(path)))
    keep_ids=[idx for idx,_ in retained]
    v=v.loc[keep_ids].copy()
    v.geometry=[candidate for _,candidate in retained]
    assert v.geometry.is_valid.all()
    partial[site]=v
    print(f'PARTIAL {site}: retained {len(v)} / '
          f'{len(v)+sum(x["site"]==site for x in partial_quarantine)}; '
          f'quarantined {sum(x["site"]==site for x in partial_quarantine)}')
assert set(partial)==set(SITES)
assert all(len(v)>0 for v in partial.values()),'Missing usable partial crowns'
# Guard against a widespread underlying geometry/reference issue.
total_partial=sum(len(v) for v in partial.values())+len(partial_quarantine)
quarantine_rate=len(partial_quarantine)/total_partial
pd.DataFrame(QA02).to_csv(OUT/'02_source_geometry_repairs.csv',index=False)
pd.DataFrame(partial_quarantine,columns=[
    'site','source_row','relative_area_change','original_valid','reason','source_path'
]).to_csv(OUT/'02_partial_quarantine.csv',index=False)
print('PARTIAL QUARANTINE:',len(partial_quarantine),'/',total_partial,
      f'({quarantine_rate:.3%})')
display(pd.DataFrame(QA02).groupby(['source','status']).agg(
    n=('row','size'),max_rel_area_change=('relative_area_change','max')).round(5))
if quarantine_rate>.01:
    raise RuntimeError('More than 1% of partial crown polygons required quarantine. '
                       'Inspect 02_partial_quarantine.csv before proceeding.')

circle_union=robust_union02(list(circles.geometry),'circle footprints')
plot_union=robust_union02(list(plots.geometry),'plot footprints')
exh_union=robust_union02(list(exh.geometry),'KVO exhaustive-positive polygons')
circ_positive=exh_union.intersection(circle_union)
plot_pos=robust_union02(list(crowns.loc[crowns.is_musanga].geometry),
                        'Nadine positive crowns').intersection(plot_union)
plot_nonpos=robust_union02(list(crowns.loc[~crowns.is_musanga].geometry),
                           'Nadine negative crowns').intersection(plot_union)
shared=circle_union.intersection(plot_union)
# Compare definitions ONLY over their shared complete-annotation footprints.
circle_pos_shared=circ_positive.intersection(shared)
plot_pos_shared=plot_pos.intersection(shared)
diff_area=float(circle_pos_shared.symmetric_difference(plot_pos_shared).area)
shared_area=float(shared.area)
plot_internal_overlap=float(plot_pos.intersection(plot_nonpos).area)
print('Circle/plot complete-annotation overlap (m²):',round(shared_area,1))
print('Musanga label disagreement on overlap (m²):',round(diff_area,1))
print('Musanga disagreement % of shared area:',
      round(100*diff_area/shared_area,3) if shared_area else 0.)
print('Plot positive vs explicit negative crown overlap (m²):',
      round(plot_internal_overlap,2))
conflicts=circle_pos_shared.symmetric_difference(plot_pos_shared)
conflicts=conflicts.union(plot_pos.intersection(plot_nonpos))
assert conflicts.is_valid
reference_summary=dict(
  schema='musanga_v4_nb02_annotation_overlap_audit_v2',
  n_circles=30,n_plots=78,n_nadine_crowns=len(crowns),
  n_exhaustive_KVO_polygons=len(exh),
  source_geometry_repair_audit=str(OUT/'02_source_geometry_repairs.csv'),
  partial_quarantine_audit=str(OUT/'02_partial_quarantine.csv'),
  n_partial_quarantined=len(partial_quarantine),
  partial_quarantine_fraction=quarantine_rate,
  circle_plot_shared_area_m2=shared_area,
  exhaustive_positive_conflict_area_m2=diff_area,
  within_plot_positive_negative_overlap_m2=plot_internal_overlap,
  circle_authoritative='V3_1_KVO_final',
  plot_authoritative='V4_Nadine_exhaustive',
  conflict_policy='suppress contradicted supervised pixels (NOT relabel)',
  plot_background_negative_assumption='PENDING MANUAL SIGNOFF: annotated=True implies exhaustive canopy Musanga annotation')
(OUT/'02_reference_annotation_overlap_audit_v2.json').write_text(json.dumps(reference_summary,indent=2))
display(pd.DataFrame([reference_summary]).T.rename(columns={0:'value'}))
print('Review geometry repairs and overlap metrics before GPU training.')


## 01B — Locate circle/plot annotation disagreements by forest

Report disagreement only on genuinely overlapping circle and plot footprints within each site. This is a **reference-consistency** analysis, not an independent accuracy assessment.

In [ ]:
# NB02.01B — Source disagreement localized to spatially overlapping site footprints.
units=GEOMS.copy()  # Initialize before Section 01B; Section 02 may reuse the same registry.
assert len(units)==108 and units.crs is not None
site_disagreement=[]
for site in SITES:
    csite=units[(units.site==site)&(units.unit_type=='circle')]
    psite=units[(units.site==site)&(units.unit_type=='plot')]
    assert len(csite) and len(psite)
    c_u=robust_union02(list(csite.geometry),f'{site}_circle_footprints')
    p_u=robust_union02(list(psite.geometry),f'{site}_plot_footprints')
    common=c_u.intersection(p_u)
    ca=circ_positive.intersection(common)
    pa=plot_pos.intersection(common)
    disagreement=ca.symmetric_difference(pa)
    both_positive=float(ca.intersection(pa).area)
    site_disagreement.append(dict(site=site,n_circles=len(csite),n_plots=len(psite),
        overlap_m2=float(common.area),circle_positive_m2=float(ca.area),
        plot_positive_m2=float(pa.area),positive_intersection_m2=both_positive,
        disagreement_m2=float(disagreement.area),
        disagreement_pct=(100*disagreement.area/common.area if common.area else np.nan),
        within_plot_label_overlap_m2=float(
            plot_pos.intersection(plot_nonpos).intersection(p_u).area)))
SITE_DISAGREEMENT=pd.DataFrame(site_disagreement)
display(SITE_DISAGREEMENT.round(3))
SITE_DISAGREEMENT.to_csv(OUT/'02_annotation_disagreement_by_site.csv',index=False)
assert np.isclose(SITE_DISAGREEMENT.overlap_m2.sum(),shared_area,atol=1.e-3),(
    'Unexpected cross-site reference overlap, inspect geometries')
assert np.isclose(SITE_DISAGREEMENT.disagreement_m2.sum(),diff_area,atol=1.e-3)
print('Annotation disagreements remain excluded from Model B supervision.')


## 02 — RGB mosaic ownership and full-chip spatial exclusion

Only RGB orthomosaics are used. Compare **all six** V3.1 inventory mosaics, including MOS 3–4 because some of the new plots may occur there. Determine each reference unit's valid *geometric* RGB mosaic coverage using its site and intersection area; do not silently drop or double-count references where mosaics overlap. The reference owner's mosaic is a deterministic maximum-footprint intersection, and coverage in ROI is reported.

Create a reproducible, bounded **256×256** candidate-chip grid for training units in each fold, then reject any candidate whose **entire image-chip footprint intersects a held-out plot/circle buffered by 50 m**, even across adjacent mosaics. This is a conservative test. Held-out units are never used as training labels, and partial crowns are also subject to this full-chip filter.

This stage is **vector/metadata only**, and outputs a sampling manifest; actual RGB reading, pixel supervision assembly, and train/validation chips come in the next step.


In [ ]:
# NB02.02 — Distinguish RGB mosaic coverage from ROI exclusion and missing data.
RGB_DIR=DATA/'Ortho_RGB'
mosaics={}
for d in V31['mosaics']:
    name=d['file']
    if not name.endswith('.tif'):continue
    p=Path(d['path'])
    if not p.exists():p=RGB_DIR/name
    if not p.exists():raise FileNotFoundError(f'Missing RGB mosaic listed in V3.1: {p}')
    with rasterio.open(p) as ds:
        assert ds.crs is not None and ds.count>=3
        bounds=gpd.GeoSeries([box(*ds.bounds)],crs=ds.crs).to_crs(REF_CRS).iloc[0]
        mosaics[name]=dict(path=p,site=str(d['site']),bounds=bounds,
                           pixel_size_m=max(abs(ds.transform.a),abs(ds.transform.e)))
assert len(mosaics)==6,[(k,v['site']) for k,v in mosaics.items()]
assert set(v['site'] for v in mosaics.values())==set(SITES)
roi=gpd.read_file(ROI_PATH).to_crs(REF_CRS)
roi_union=robust_union02(list(roi.geometry),'general RGB ROI')
units=GEOMS.copy()
owners=[];all_options=[]
for u in units.itertuples(index=False):
    options=[]
    for name,d in mosaics.items():
        if d['site']!=u.site:continue
        footprint_ha=float(u.geometry.intersection(d['bounds']).area)/1e4
        roi_area_ha=float(u.geometry.intersection(d['bounds']).intersection(roi_union).area)/1e4
        area=max(float(u.geometry.area)/1e4,1.e-12)
        options.append(dict(unit_type=str(u.unit_type),unit_id=str(u.unit_id),
            site=str(u.site),fold=int(u.fold),mosaic=name,
            raw_RGB_bbox_coverage=footprint_ha/area,
            RGB_and_ROI_bbox_coverage=roi_area_ha/area))
    assert options
    all_options.extend(options)
    # Prefer the largest RGB+ROI geometric overlap; never accept zero-covered
    # assignments as valid merely because they are deterministic.
    best=sorted(options,key=lambda q:(q['RGB_and_ROI_bbox_coverage'],
                                     q['raw_RGB_bbox_coverage'],q['mosaic']),reverse=True)[0]
    owners.append(dict(unit_type=str(u.unit_type),unit_id=str(u.unit_id),
       site=str(u.site),fold=int(u.fold),spatial_group=str(u.spatial_group),
       mosaic=best['mosaic'],
       rgb_bbox_coverage_fraction=best['raw_RGB_bbox_coverage'],
       roi_mosaic_coverage_fraction=best['RGB_and_ROI_bbox_coverage']))
OWNERS=pd.DataFrame(owners)
OPTIONS=pd.DataFrame(all_options)
assert not OWNERS.duplicated(['unit_type','unit_id']).any()
print('RGB mosaics assigned to each site/reference type:')
display(OWNERS.groupby(['site','unit_type','mosaic']).agg(n=('unit_id','size'),
   min_roi_coverage=('roi_mosaic_coverage_fraction','min')).round(4))
PROBLEMS=OWNERS[OWNERS.roi_mosaic_coverage_fraction.lt(.95)].copy()
print('References with <95% geometric RGB + ROI coverage:',len(PROBLEMS))
display(PROBLEMS)
if len(PROBLEMS):
    print('All candidate mosaics for coverage-problem units:')
    display(OPTIONS.merge(PROBLEMS[['unit_type','unit_id']],on=['unit_type','unit_id'],
        validate='many_to_one').sort_values(['unit_id','RGB_and_ROI_bbox_coverage'],
        ascending=[True,False]).round(5))
    print('Interpretation: nonzero raw_RGB_bbox_coverage with zero RGB_and_ROI_bbox_coverage '
          'indicates ROI exclusion, not necessarily missing RGB data.')
OWNERS.to_csv(OUT/'02_reference_mosaic_owners_v2.csv',index=False)
OPTIONS.to_csv(OUT/'02_all_reference_mosaic_coverage_options.csv',index=False)
PROBLEMS.to_csv(OUT/'02_unresolved_rgb_coverage_units.csv',index=False)
# Only zero-coverage or low-coverage reference units are marked unavailable;
# preserve their NB01 fold assignments for provenance and spatial buffering.
UNAVAILABLE_KEYS=set(zip(PROBLEMS.unit_type.astype(str),PROBLEMS.unit_id.astype(str)))
assert set(zip(PROBLEMS.unit_type.astype(str),PROBLEMS.unit_id.astype(str)))=={('plot','Mixt1'),('plot','Mixt2')},(
    'Unexpected reference availability; investigate before freezing exclusions',PROBLEMS[['unit_type','unit_id']])
EXCLUSIONS=PROBLEMS[['unit_type','unit_id','site','fold','spatial_group','mosaic',
                     'rgb_bbox_coverage_fraction','roi_mosaic_coverage_fraction']].copy()
EXCLUSIONS['reason']='zero_RGB_bbox_intersection_in_all_site_mosaics'
EXCLUSIONS['eligible_training']=False
EXCLUSIONS['eligible_evaluation']=False
EXCLUSIONS.to_csv(OUT/'02_explicit_reference_exclusions.csv',index=False)
print('Fixed exclusion registry (fold geometry remains held out):')
display(EXCLUSIONS)
# Full-chip holdout applies to all reference units, across all mosaics.
HELDOUT={}
for f in range(3):
    q=units.loc[units.fold.eq(f)]
    HELDOUT[f]={site:robust_union02(
        [geom.buffer(50.) for geom in q.loc[q.site.eq(site),'geometry']],
        f'holdout_{f}_{site}') for site in SITES}
assert all(len(units.loc[units.fold.eq(f)])>0 for f in range(3))
print('Holdout buffers prepared. Unresolved low-coverage references cannot silently enter training.')


## 03 — Candidate training-chip manifest (geometry only; no image read)

Generate deterministic non-overlapping training-chip candidates from outer-training circles and outer-training plots, plus partial crown centres. The manifest records chip width, source unit/group, site and whether the candidate is **fully clear** of every held-out reference plus 50 m. Candidate source attribution remains separate from pixel-level label reconciliation. Sampling caps prevent excessive Drive I/O.

For Model A, plots are never training sources. For Model B, plot-labelled candidates are added. **Both use the same baseline circle and partial-crown candidates**, making training differences interpretable.


In [ ]:
CHIP=256
MAX_CANDIDATES_PER_UNIT=40
MAX_PARTIAL_PER_SITE=240
SEED=20261118
from rasterio.windows import Window
def candidate_chip_centres(ds,geom,max_chips):
    # Build a bounded grid of chip centers inside source geometry. The chip itself
    # may protrude; full-window holdout exclusion is checked separately.
    in_ds=gpd.GeoSeries([geom],crs=REF_CRS).to_crs(ds.crs).iloc[0]
    raw=rasterio.windows.from_bounds(*in_ds.bounds,transform=ds.transform)
    step=CHIP
    rmin=max(CHIP//2,int(np.floor(raw.row_off)))
    cmin=max(CHIP//2,int(np.floor(raw.col_off)))
    rmax=min(ds.height-CHIP//2,int(np.ceil(raw.row_off+raw.height)))
    cmax=min(ds.width-CHIP//2,int(np.ceil(raw.col_off+raw.width)))
    if rmax<rmin or cmax<cmin:return []
    rrange=list(range(rmin,rmax+1,step)) or [rmin]
    crange=list(range(cmin,cmax+1,step)) or [cmin]
    pts=[]
    for rr in rrange:
        for cc in crange:
            xy=ds.xy(rr,cc)
            point=gpd.GeoSeries(gpd.points_from_xy([xy[0]],[xy[1]]),crs=ds.crs).to_crs(REF_CRS).iloc[0]
            if geom.intersects(point):pts.append((rr,cc))
    # Always offer centroid so narrow or sub-chip-sized units yield a candidate.
    xy=in_ds.representative_point()
    row,col=ds.index(xy.x,xy.y)
    if CHIP//2<=row<ds.height-CHIP//2 and CHIP//2<=col<ds.width-CHIP//2:
        pts.append((row,col))
    pts=sorted(set(pts))
    if len(pts)>max_chips:
        take=np.linspace(0,len(pts)-1,max_chips).astype(int)
        pts=[pts[i] for i in take]
    return pts
records=[]
for fold in range(3):
    fold_units=units[units.fold.ne(fold)].copy()
    for mosaic,d in mosaics.items():
        site=d['site']
        heldout=HELDOUT[fold][site]
        assigned=OWNERS[(OWNERS.mosaic==mosaic)&OWNERS.fold.ne(fold)&
                        OWNERS.roi_mosaic_coverage_fraction.ge(.95)]
        with rasterio.open(d['path']) as ds:
            for unit in assigned.itertuples(index=False):
                geom=fold_units[(fold_units.unit_type==unit.unit_type)&
                    (fold_units.unit_id==unit.unit_id)].geometry.iloc[0]
                for rr,cc in candidate_chip_centres(ds,geom,MAX_CANDIDATES_PER_UNIT):
                    win=Window(int(cc-CHIP//2),int(rr-CHIP//2),CHIP,CHIP)
                    bbox=box(*rasterio.windows.bounds(win,ds.transform))
                    bb=gpd.GeoSeries([bbox],crs=ds.crs).to_crs(REF_CRS).iloc[0]
                    records.append(dict(fold=fold,site=site,mosaic=mosaic,
                       row=int(win.row_off),col=int(win.col_off),
                       source='circle' if unit.unit_type=='circle' else 'plot',
                       source_id=str(unit.unit_id),spatial_group=str(unit.spatial_group),
                       blocked_by_outer_holdout=bool(bb.intersects(heldout)),
                       full_chip_area_m2=float(bb.area)))
            # Sample explicit partial crowns only outside holdout buffers.
            crown=partial[site]
            if len(crown):
                z=crown[crown.geometry.intersects(d['bounds'])].copy()
                # Stable spatial candidate selection, no held-out labels consulted.
                if len(z)>MAX_PARTIAL_PER_SITE:
                    z=z.sample(n=MAX_PARTIAL_PER_SITE,random_state=SEED+fold)
                for ix,part in z.iterrows():
                    xy=part.geometry.representative_point()
                    r,c=ds.index(xy.x,xy.y)
                    win=Window(int(c-CHIP//2),int(r-CHIP//2),CHIP,CHIP)
                    if (win.row_off<0 or win.col_off<0 or
                        win.col_off+CHIP>ds.width or win.row_off+CHIP>ds.height):continue
                    bb=gpd.GeoSeries([box(*rasterio.windows.bounds(win,ds.transform))],
                                     crs=ds.crs).to_crs(REF_CRS).iloc[0]
                    records.append(dict(fold=fold,site=site,mosaic=mosaic,
                       row=int(win.row_off),col=int(win.col_off),source='partial',
                       source_id=str(ix),spatial_group='partial',
                       blocked_by_outer_holdout=bool(bb.intersects(heldout)),
                       full_chip_area_m2=float(bb.area)))
CANDIDATES=pd.DataFrame(records)
assert not CANDIDATES.empty
# A mosaic/window may overlap several original annotations: keep the first in
# each source class for source balancing; labels will be constructed from geometry.
CANDIDATES=CANDIDATES.drop_duplicates(['fold','site','mosaic','row','col','source']).copy()
CANDIDATES['eligible']=~CANDIDATES.blocked_by_outer_holdout
print('Candidate windows, by fold/source and leakage status:')
display(CANDIDATES.groupby(['fold','source','eligible']).size().unstack(fill_value=0))
print('Candidate mosaic attribution:')
display(CANDIDATES.loc[CANDIDATES.eligible].groupby(['fold','site','source']).size().unstack(fill_value=0))
if CANDIDATES.loc[CANDIDATES.eligible].empty:
    raise RuntimeError('No training candidates after holdout exclusion')
assert not CANDIDATES.loc[CANDIDATES.eligible].blocked_by_outer_holdout.any()
assert not any((str(r.source),str(r.source_id)) in UNAVAILABLE_KEYS for r in
    CANDIDATES.loc[CANDIDATES.eligible].itertuples(index=False)), 'Uncovered plot was sampled'
CANDIDATES.to_csv(OUT/'02_full_chip_heldout_exclusion_manifest.csv',index=False)
print('Manifest:',OUT/'02_full_chip_heldout_exclusion_manifest.csv')
print('Important: this is a candidate manifest, NOT a guarantee of supervised pixel counts.')


## 04 — Paired pixel supervision audit (CPU; baseline must be identical)

Read a limited sample of actual 256×256 RGB chips. **Model A and B must have identical positive/negative labels and weights on every baseline-supervised pixel.** Model B may only add labels on pixels with no baseline supervision (within training-fold plot footprints, and never in a circle), and conflicting plot/partial labels in the added support remain unknown.

The plot-negative assumption remains **provisional** until verified. We audit label counts and generate a deterministic pairwise equality test at pixel resolution, not merely differences between aggregated positive-pixel totals. Neither model is trained in this notebook.


In [ ]:
# NB02.04 — Bounded true RGB window reads, with source-specific pixel masks.
# Small sample of candidate windows; this is NOT yet the production training cache.
from rasterio.features import rasterize
from shapely.geometry import box as shape_box
from rasterio.windows import Window
from collections import defaultdict
AUDIT_PER_SITE_SOURCE_FOLD=3
CHIP_AUDIT_SEED=20261118
SOURCE_CLASSES=['circle','plot','partial']
audit_recs=[]
audit_selected=[]
for (fold,site,source),frame in CANDIDATES.loc[CANDIDATES.eligible].groupby(
        ['fold','site','source'],sort=True):
    take=min(AUDIT_PER_SITE_SOURCE_FOLD,len(frame))
    sample=frame.sample(n=take,random_state=CHIP_AUDIT_SEED+100*int(fold)+SITES.index(site)*7+SOURCE_CLASSES.index(source))
    audit_selected.append(sample)
AUDIT_WINDOWS=pd.concat(audit_selected,ignore_index=True)
assert len(AUDIT_WINDOWS)<=3*len(SITES)*len(SOURCE_CLASSES)*AUDIT_PER_SITE_SOURCE_FOLD
print('Audit window counts:',AUDIT_WINDOWS.groupby(['fold','site','source']).size().to_dict())
# A/B share the EXACT circle and partial training candidates.
# Annotation masks always exclude the outer fold's complete references.
def burn_geom(geom,win,ds):
    if geom is None or geom.is_empty:return np.zeros((CHIP,CHIP),dtype=bool)
    window_box=shape_box(*rasterio.windows.bounds(win,ds.transform))
    if not geom.intersects(window_box):return np.zeros((CHIP,CHIP),dtype=bool)
    clipped=geom.intersection(window_box)
    if clipped.is_empty:return np.zeros((CHIP,CHIP),dtype=bool)
    return rasterize([(clipped,1)],out_shape=(CHIP,CHIP),
           transform=rasterio.windows.transform(win,ds.transform),
           fill=0,dtype='uint8').astype(bool)
def burn_geoms(geometries,win,ds):
    window_box=shape_box(*rasterio.windows.bounds(win,ds.transform))
    inter=[(z.intersection(window_box),1) for z in geometries if z is not None and z.intersects(window_box)]
    if not inter:return np.zeros((CHIP,CHIP),dtype=bool)
    return rasterize(inter,out_shape=(CHIP,CHIP),
        transform=rasterio.windows.transform(win,ds.transform),
        fill=0,dtype='uint8').astype(bool)
# Raster training only needs the active training-group footprints; held-out
# reference positive crowns CANNOT leak into target masks.
TRAIN_REFS={}
for f in range(3):
    train=units[units.fold.ne(f)]
    TRAIN_REFS[f]={}
    for site in SITES:
        cs=robust_union02(list(train[(train.site==site)&(train.unit_type=='circle')].geometry),
                          f'train_circle_f{f}_{site}')
        ps=robust_union02(list(train[(train.site==site)&(train.unit_type=='plot')].geometry),
                          f'train_plot_f{f}_{site}')
        TRAIN_REFS[f][site]=(cs,ps)
# Precompute fast spatial index on the optional partial crowns.
# Only explicit crown geometry is allowed to supervise the partial area.
for (f,site,mosaic),q in AUDIT_WINDOWS.groupby(['fold','site','mosaic']):
    ds_path=mosaics[mosaic]['path']
    c_train,p_train=TRAIN_REFS[int(f)][site]
    site_partial=partial[site]
    for row in q.itertuples(index=False):
        win=Window(int(row.col),int(row.row),CHIP,CHIP)
        with rasterio.open(ds_path) as ds:
            native_box=shape_box(*rasterio.windows.bounds(win,ds.transform))
            spatial_box=gpd.GeoSeries([native_box],crs=ds.crs).to_crs(REF_CRS).iloc[0]
            assert not spatial_box.intersects(HELDOUT[int(f)][site]),'Holdout chip leakage'
            rgb=ds.read([1,2,3],window=win,masked=True)
            valid=~np.any(np.ma.getmaskarray(rgb),axis=0)
            if ds.count>=4:
                alpha=ds.read(4,window=win,masked=True)
                valid &= (~np.ma.getmaskarray(alpha)) & (np.asarray(alpha.filled(0))>0)
            valid &= np.all(np.isfinite(np.asarray(rgb.data,dtype=np.float32)),axis=0)
            valid &= burn_geom(roi_union,win,ds)
            c_mask=burn_geom(c_train,win,ds)
            p_mask=burn_geom(p_train,win,ds)
            c_pos=burn_geom(circ_positive,win,ds)&c_mask
            p_pos=burn_geom(plot_pos,win,ds)&p_mask
            p_negative_explicit=burn_geom(plot_nonpos,win,ds)&p_mask
            internal_conflict=burn_geom(conflicts,win,ds)
            # Queries on original site-specific partial annotations; no background negatives.
            hits=site_partial.sindex.query(spatial_box,predicate='intersects')
            subset=site_partial.iloc[hits]
            ptrees=burn_geoms(list(subset.loc[subset.musanga].geometry),win,ds)
            ntrees=burn_geoms(list(subset.loc[~subset.musanga].geometry),win,ds)
            # Shared BASELINE (both arms): exhaustive circle + explicit partial crowns.
            # Circle pixels are authoritative. Do not supervise contradictory
            # circle/partial or partial+/partial- pixels in either arm.
            partial_conflict=(ptrees&ntrees) | (ptrees&c_mask&~c_pos) | (ntrees&c_pos)
            a_pos=c_pos|(~c_mask&ptrees&~ntrees)
            a_weight=(c_mask|((ptrees|ntrees)&~c_mask))&~partial_conflict
            a_weight &=valid
            # Model B adds only previously UNKNOWN plot-exclusive pixels.
            # No B plot information is permitted to flip, unmask or suppress
            # an A-supervised pixel. The original A and B baseline masks match.
            extra_plot=p_mask&~c_mask&~a_weight
            plot_new_conflict=(p_pos&p_negative_explicit) | (
                 (ptrees&~p_pos)|(ntrees&p_pos))
            added_weight=extra_plot&~plot_new_conflict&valid
            added_positive=added_weight&p_pos
            b_pos=a_pos|added_positive
            b_weight=a_weight|added_weight
            assert np.array_equal(a_weight&valid,b_weight&a_weight&valid)
            assert np.array_equal(a_pos[a_weight],b_pos[a_weight]),(
                'Model B changed original Model A labels')
            assert np.all((b_weight&~a_weight)<=extra_plot)
            assert not np.any(added_weight&c_mask)
            assert not np.any(added_weight&burn_geom(HELDOUT[int(f)][site],win,ds))
            # All native-image pixels outside available RGB/ROI stay unknown.
            for model,target,weight in [('A',a_pos,a_weight),('B',b_pos,b_weight)]:
                weight=weight&valid
                positives=int((weight&target).sum())
                negatives=int((weight&~target).sum())
                audit_recs.append(dict(fold=int(f),site=site,mosaic=mosaic,
                    sample_source=str(row.source),sample_source_id=str(row.source_id),
                    model=model,row=int(row.row),col=int(row.col),pixels=CHIP*CHIP,
                    valid_RGB_ROI_pixels=int(valid.sum()),
                    supervised_positive_pixels=positives,
                    supervised_negative_pixels=negatives,
                    unknown_or_invalid_pixels=int(CHIP*CHIP-positives-negatives),
                    circle_support_pixels=int((c_mask&valid).sum()),
                    plot_only_support_pixels=int((extra_plot&valid).sum()),
                    masked_label_conflict_pixels=int((internal_conflict&valid).sum()),
                    baseline_supervised_pixels=int(a_weight.sum()),
                    B_added_plot_pixels=int(added_weight.sum()) if model=='B' else 0,
                    supervised_total_pixels=positives+negatives))
AUDIT_PIXELS=pd.DataFrame(audit_recs)
assert len(AUDIT_PIXELS)==2*len(AUDIT_WINDOWS)
assert AUDIT_PIXELS.groupby(['fold','site','sample_source','model']).size().min()>0
AUDIT_PIXELS.to_csv(OUT/'02_sampled_RGB_pixel_supervision_v2.csv',index=False)
print('Sampled RGB pixel supervision, per fold/model/source (counts in pixels):')
display(AUDIT_PIXELS.groupby(['fold','model','sample_source'])[
  ['valid_RGB_ROI_pixels','supervised_positive_pixels','supervised_negative_pixels',
   'unknown_or_invalid_pixels','masked_label_conflict_pixels']
].sum().round(0))
print('Per-site supervised support:')
display(AUDIT_PIXELS.groupby(['site','model'])[
    ['supervised_positive_pixels','supervised_negative_pixels','supervised_total_pixels']
].sum().round(0))
print('Audited individual windows with 0 useful supervised pixels:',
      int((AUDIT_PIXELS.supervised_total_pixels==0).sum()),'of',len(AUDIT_PIXELS))
PAIRS=AUDIT_PIXELS.pivot(index=['fold','site','mosaic','sample_source','sample_source_id','row','col'],
                             columns='model',values=['supervised_total_pixels','baseline_supervised_pixels'])
assert np.array_equal(PAIRS[('baseline_supervised_pixels','A')].to_numpy(),
                      PAIRS[('baseline_supervised_pixels','B')].to_numpy())
assert (PAIRS[('supervised_total_pixels','B')]>=PAIRS[('supervised_total_pixels','A')]).all()
print('PAIRWISE LABEL INVARIANT: every Model A supervised pixel and label is retained in Model B.')
print('B added plot-only supervised pixels:',int(AUDIT_PIXELS.loc[AUDIT_PIXELS.model=='B','B_added_plot_pixels'].sum()))
print('WARNING: Model B plot background is provisional pending exhaustive-negative sign-off.')


## 05 — Frozen paired candidate manifest and source-balance audit

Both arms use the **same, deterministic baseline candidate windows** (circle and partial), and Model B additionally uses plot-source candidate windows. This is a *candidate-window manifest*, not yet the final RGB-valid training cache. Rows are uniquely keyed, contain holdout/fold provenance, and the 2 uncovered MOS plots are explicitly excluded. Counts and sampling targets are reported by site, fold and source so that the later GPU training can use a common per-epoch baseline budget plus a controlled plot supplement.

A paired comparison of equal *epoch count* alone is not equal computational budget: training must fix baseline exposure and account for additional plot examples. The subsequent training notebook should use the same pixel-label generator as this preflight, and rebuild full RGB caches with validation checks.


In [ ]:
# Candidate manifest: identical A/B baseline rows and optional plot additions.
eligible=CANDIDATES[CANDIDATES.eligible].copy()
eligible['source_id']=eligible.source_id.astype(str)
eligible['window_uid']=eligible.apply(lambda z:
    f"f{int(z.fold)}|{z.site}|{z.mosaic}|r{int(z.row)}c{int(z.col)}|{z.source}",axis=1)
assert eligible.window_uid.is_unique
assert not any((str(z.source),str(z.source_id)) in UNAVAILABLE_KEYS
               for z in eligible.itertuples(index=False))
base=eligible[eligible.source.isin(['circle','partial'])].copy()
plots_extra=eligible[eligible.source.eq('plot')].copy()
assert len(base)>0 and len(plots_extra)>0
armA=base.assign(model='A')
armB=pd.concat([base.assign(model='B'),plots_extra.assign(model='B')],ignore_index=True)
PAIRED_CANDIDATES=pd.concat([armA,armB],ignore_index=True).sort_values(
    ['fold','site','model','source','window_uid']).reset_index(drop=True)
assert not PAIRED_CANDIDATES.duplicated(['model','window_uid']).any()
set_A=set(armA.window_uid);set_B_baseline=set(armB.loc[armB.source.isin(['circle','partial']),'window_uid'])
assert set_A==set_B_baseline
assert set(armB.loc[armB.source.eq('plot'),'window_uid']).isdisjoint(set_A)
assert not PAIRED_CANDIDATES.blocked_by_outer_holdout.any()
PAIRED_CANDIDATES.to_csv(OUT/'02_paired_model_training_window_candidates_v3.csv',index=False)
CANDIDATE_BALANCE=PAIRED_CANDIDATES.groupby(['fold','site','model','source']).size().unstack(fill_value=0).reset_index()
display(CANDIDATE_BALANCE)
CANDIDATE_BALANCE.to_csv(OUT/'02_candidate_balance_by_site_fold.csv',index=False)
print('Model A candidate windows:',len(armA),'Model B:',len(armB),
      'B-only plot windows:',len(plots_extra))
print('IMPORTANT: this is NOT the complete train cache; RGB/ROI valid pixel '
      'coverage and label support require per-window acceptance in the next stage.')


## 06 — Freeze CPU audit and training eligibility contract

Explicitly preserve original NB01 folds, document excluded non-imaged MOS plots, verify sampled pixel-level A/B baseline invariance and publish deterministic paired window-candidate manifests. GPU training remains disabled until the plot-background assumption is approved and production chip-label caches have been checked.


In [ ]:
# Freeze a CPU-only gate report; explicit 'NOT TRAIN-READY' status prevents accidental use.
assert len(GEOMS)==108 and GEOMS.spatial_group.nunique()>0
assert len(OWNERS)==108
assert CANDIDATES.eligible.any()
assert set(CANDIDATES.source).issubset({'circle','plot','partial'})
assert CANDIDATES.groupby('fold').eligible.sum().gt(0).all()
train_counts=CANDIDATES.loc[CANDIDATES.eligible].groupby(['fold','source']).size().unstack(fill_value=0)
assert all(k in train_counts.columns for k in ['circle','plot','partial']),train_counts.columns
assert train_counts[['circle','plot','partial']].min().min()>0,'Insufficient sources in at least one fold'
manifest=dict(schema='musanga_v4_nb02_cpu_pixel_preflight_v3',
  status='CPU_PAIRED_BASELINE_AUDIT_COMPLETE_GPU_TRAINING_NOT_ENABLED',
  nb01_csv=str(FOLD_CSV),nb01_csv_sha256=SPLIT_CONTRACT['csv_sha256'],
  original_circle_folds_preserved=True,
  unresolved_reference_units=PROBLEMS[['unit_type','unit_id','site','roi_mosaic_coverage_fraction']].to_dict('records'),
  training_excluded_reference_units=EXCLUSIONS[['unit_type','unit_id','site','fold','reason']].to_dict('records'),
  usable_reference_units=106,
  usable_evaluation_circles=30,usable_evaluation_plots=76,
  sampled_rgb_pixel_audit=str(OUT/'02_sampled_RGB_pixel_supervision_v2.csv'),
  site_annotation_disagreement=str(OUT/'02_annotation_disagreement_by_site.csv'),
  n_reference_units=108,n_spatial_groups=int(GEOMS.spatial_group.nunique()),
  n_folds=3,full_chip_px=CHIP,holdout_buffer_m=50,
  model_A='KVO_complete_circles_plus_partial_tree_crowns',
  model_B='Model_A_plus_Nadine_exhaustive_field_plots',
  sampling='identical baseline window list and baseline supervision; B adds plot-exclusive unknown pixels only',
  baseline_A_B_pixel_invariance='PASS on sampled RGB windows',
  paired_training_window_manifest=str(OUT/'02_paired_model_training_window_candidates_v3.csv'),
  paired_training_window_manifest_sha256=hashlib.sha256(
      (OUT/'02_paired_model_training_window_candidates_v3.csv').read_bytes()).hexdigest(),
  explicit_pending_checks=[
    'Mixt1/Mixt2 are omitted from training and scoring until corrected RGB footprints exist',
    'Approve plot unannotated background as negative or restrict to crown objects',
    'Confirm conservative conflict masking and A/B baseline equality on complete production caches',
    'Review sampled RGB positive/negative supervision before generating full train caches',
    'Enforce full-chip holdout filtering on final train AND inner selection',
    'Restrict plot and circle annotations to outer training groups',
    'Fix 256px model context and harmonized augmentation/optimizer',
    'Choose inner spatial selection without outer-fold outcome tuning'],
  candidates_csv=str(OUT/'02_full_chip_heldout_exclusion_manifest.csv'),
  candidates_sha256=hashlib.sha256((OUT/'02_full_chip_heldout_exclusion_manifest.csv').read_bytes()).hexdigest(),
  reference_overlap_audit=str(OUT/'02_reference_annotation_overlap_audit_v2.json'))
(OUT/'02_cpu_gate_contract.json').write_text(json.dumps(manifest,indent=2))
display(train_counts)
print('NB02 CPU RGB PIXEL AUDIT COMPLETE: GPU TRAINING INTENTIONALLY NOT ENABLED')
print('Saved:',OUT/'02_cpu_gate_contract.json')
print('Review outputs before generating supervised-pixel caches and training checkpoints.')
